In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/all_results.csv')

In [10]:
print(df.columns.tolist())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [11]:
df = df[['id', 'time_window_start', 'username', 'Demo_sex_Category', 'Demo_age_Category', 'Demo_ethnicity_Category', 'Demo_highest_educational_qualification_Category','Demo_past_vote_vote_choice_in_the_2020_presidential_election_Category', 'Demo_interest_in_politics_Category', 'Demo_marital_status_Category', 'Demo_partisan_loyalties_Category','Demo_which_state_of_the_usa_do_they_live_in__Category','Demo_household_income_bracket_Category','Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Category']]

In [ ]:
df = df[['id', 'time_window_start', 'username', 'Demo_sex_Symbol', 'Demo_age_Symbol', 'Demo_ethnicity_Category', 'Demo_highest_educational_qualification_Category','Demo_past_vote_vote_choice_in_the_2020_presidential_election_Category', 'Demo_interest_in_politics_Category', 'Demo_marital_status_Category', 'Demo_partisan_loyalties_Category','Demo_which_state_of_the_usa_do_they_live_in__Category','Demo_household_income_bracket_Category','Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Category']]

In [15]:
df_clean = df.dropna()

In [ ]:
values_to_remove = ['not from a state in the usa', 'usa']
df_filtered = df_clean[~df_clean['Demo_which_state_of_the_usa_do_they_live_in__Category'].isin(values_to_remove)]

In [19]:
df_filtered.to_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/abortion_result.csv')

In [3]:
df_filtered = pd.read_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/abortion_result.csv')

In [78]:
df_filtered['Demo_marital_status_Category'].unique()

array(['married currently legally married and living with a spouse',
       'single never married, including those who are legally separated',
       'divorced legally divorced and not currently remarried',
       'single never married',
       'single never married, including those legally separated',
       'single never married (including separated)', 'single',
       'single never married, including those who are legally separated,',
       'married', 'single never married, including legally separated',
       'widowed spouse has passed away and not currently remarried',
       'widowed',
       'single never married, including those who are legally separated.',
       'single never married (including legally separated)',
       'married currently legally married and living with a spouse,',
       'single never married or legally separated',
       'married currently legally married and living with a spouse.',
       'single never married (or legally separated)'], dtype=object)

In [81]:
import pandas as pd
import numpy as np

def clean_marital_status(status):
    """Clean marital status into standardized categories"""
    if pd.isna(status):
        return status
    
    status_str = str(status).lower().strip()
    
    # Map to standardized categories
    if any(term in status_str for term in ['married', 'spouse']):
        return 'Married'
    elif any(term in status_str for term in ['single', 'never married']):
        return 'Single'
    elif 'divorced' in status_str:
        return 'Divorced'
    elif 'widowed' in status_str:
        return 'Widowed'
    else:
        return status  

# Apply the cleaning
df_filtered['marital_status_cleaned'] = df_filtered['Demo_marital_status_Category'].apply(clean_marital_status)
print(df_filtered['marital_status_cleaned'].value_counts())

marital_status_cleaned
Married    8297
Single      125
Widowed      15
Name: count, dtype: int64


In [74]:
import pandas as pd
import numpy as np

def clean_politics_interest(interest):
    """Clean politics interest into standardized categories"""
    if pd.isna(interest):
        return interest
    
    interest_str = str(interest).lower().strip()
    
    # Remove punctuation and common phrases
    interest_str = (interest_str
                   .replace(' in politics', '')
                   .replace(' at politics', '')
                   .replace(' at all', '')
                   .replace(',', '')
                   .replace('.', '')
                   .replace(')', '')
                   .strip())
    
    # Map to standardized categories
    if 'highly' in interest_str:
        return 'Highly interested'
    elif 'moderately' in interest_str:
        return 'Moderately interested'
    elif 'slightly' in interest_str:
        return 'Slightly interested'
    elif 'not interested' in interest_str:
        return 'Not interested'
    else:
        return interest 

# Apply the cleaning
df_filtered['politics_interest_cleaned'] = df_filtered['Demo_interest_in_politics_Category'].apply(clean_politics_interest)
print(df_filtered['politics_interest_cleaned'].value_counts())

politics_interest_cleaned
Highly interested        5956
Moderately interested    1530
Slightly interested       622
Not interested            329
Name: count, dtype: int64


In [67]:
import pandas as pd
import re

def clean_vote_choice(value: str) -> str:
    """Normalize messy 2020 presidential vote responses."""
    if pd.isna(value):
        return None

    v = value.strip().lower()

    # Remove punctuation and excess whitespace
    v = re.sub(r'[.,]', '', v)
    v = re.sub(r'\s+', ' ', v)

    # --- Standardized categories ---
    if 'did not vote' in v or 'didn’t vote' in v or 'didnt vote' in v:
        return 'Did not vote'

    elif 'trump' in v:
        return 'Trump'

    elif 'biden' in v:
        return 'Biden'

    elif 'jorgensen' in v or 'libertarian' in v:
        return 'Jorgensen'

    elif 'hawkins' in v or 'green party' in v:
        return 'Hawkins'

    elif 'other' in v:
        return 'Other'

    return v

df_filtered['vote_cleaned'] = df_filtered['Demo_past_vote_vote_choice_in_the_2020_presidential_election_Category'].apply(clean_vote_choice)

# Check results
print(df_filtered['vote_cleaned'].value_counts())

vote_cleaned
Trump           3825
Biden           3013
Did not vote    1454
Jorgensen        132
Hawkins           13
Name: count, dtype: int64


In [41]:
import pandas as pd
import numpy as np

def clean_gender_robust(gender):
    if pd.isna(gender):
        return gender
    
    gender_str = str(gender).lower().strip()
    
    # Direct mapping for common patterns
    gender_mapping = {
        # Female patterns
        'feminine sex female': 'Female',
        'feminine sex female.': 'Female',
        'feminine sex female,': 'Female',
        'feminine sex female)': 'Female',
        'feminine': 'Female',
        'feminine sex': 'Female',
        'feminine sex male': 'Female',  # Assuming this is a typo, feminine should be female
        
        # Male patterns  
        'masculine sex male': 'Male',
        'masculine sex male,': 'Male',
        'masculine sex male.': 'Male',
        'masculine sex': 'Male',
        'masculine': 'Male',
        'male': 'Male'
    }
    
    # Check for exact matches first
    if gender_str in gender_mapping:
        return gender_mapping[gender_str]
    
    # Check for partial matches
    if any(female_term in gender_str for female_term in ['female', 'feminine']):
        return 'Female'
    elif any(male_term in gender_str for male_term in ['male', 'masculine']):
        return 'Male'
    
    return gender  # Return original for unknown cases


df_filtered['gender_cleaned'] = df_filtered['Demo_sex_Category'].apply(clean_gender_robust)

print(df_filtered['gender_cleaned'].value_counts())

gender_cleaned
Male      5460
Female    2977
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np

def clean_education_two_categories(education):
    if pd.isna(education):
        return education
    
    education_str = str(education).lower().strip()
    
    # Q1: High School level education (including vocational training and associate degree)
    high_school_indicators = [
        'high school',
        'vocational training', 
        'associate degree',
        'vocational',
        'associate'
    ]
    
    # Q2: College/University level education (including bachelor's, master's, doctorate)
    college_indicators = [
        'college',
        'university', 
        "bachelor's",
        'bachelors',
        "master's", 
        'masters',
        'doctorate',
        'md)',
        'bachelor',
        'master',
        'collegelevel'
    ]
    
    # Check for Q1 indicators first
    if any(indicator in education_str for indicator in high_school_indicators):
        return 'Q1) completed education up to and including high school - high school diploma, vocational training, associate degree'
    
    # Check for Q2 indicators
    elif any(indicator in education_str for indicator in college_indicators):
        return 'Q2) completed education at the college or university level - bachelor\'s degree, master\'s degree, doctorate'
    
    # # Fallback for any edge cases - default to Q2 if it contains educational terms
    # elif any(term in education_str for term in ['education', 'degree', 'diploma']):
    #     return 'Q2) completed education at the college or university level - bachelor\'s degree, master\'s degree, doctorate'
    
    else:
        # Return original for manual inspection if no patterns match
        return education

df_filtered['education_cleaned'] = df_filtered['Demo_highest_educational_qualification_Category'].apply(clean_education_two_categories)

print(df_filtered['education_cleaned'].value_counts())


education_cleaned
Q2) completed education at the college or university level - bachelor's degree, master's degree, doctorate              6346
Q1) completed education up to and including high school - high school diploma, vocational training, associate degree    2091
Name: count, dtype: int64


In [60]:
df_filtered['education_cleaned'] = df_filtered['education_cleaned'].replace({
    'Q1) completed education up to and including high school - high school diploma, vocational training, associate degree': 'High School',
    'Q2) completed education at the college or university level - bachelor\'s degree, master\'s degree, doctorate': 'University'
})

In [5]:
import re

def clean_belief(x):
    """
    Normalize abortion opinion strings into four ordered categories:
    strongly_disagree, somewhat_disagree, somewhat_agree, strongly_agree
    """
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()
    x = re.sub(r'[",.\']', '', x)  # remove punctuation and quotes

    if "strongly disagree" in x or "strongly opposes" in x:
        return "strongly_disagree"
    elif "somewhat disagree" in x:
        return "somewhat_disagree"
    elif "somewhat agree" in x:
        return "somewhat_agree"
    elif "strongly agree" in x:
        return "strongly_agree"
    elif "agree" in x and not ("disagree" in x):
        # catch short variants like 'strongly agree', 'somewhat agree'
        if "strongly" in x:
            return "strongly_agree"
        elif "somewhat" in x:
            return "somewhat_agree"
        else:
            return "somewhat_agree"
    elif "disagree" in x and not ("agree" in x):
        if "strongly" in x:
            return "strongly_disagree"
        elif "somewhat" in x:
            return "somewhat_disagree"
        else:
            return "somewhat_disagree"
    else:
        return np.nan

df_filtered["belief_clean"] = df_filtered['Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Category'].apply(clean_belief)
print(df_filtered["belief_clean"].value_counts(dropna=False))


belief_clean
strongly_disagree    3801
strongly_agree       3688
somewhat_agree        834
somewhat_disagree     140
Name: count, dtype: int64


In [11]:


def clean_ethnicity(x):
    """
    Normalize ethnicity categories into the standardized set:
    white, black_or_african_american, hispanic_or_latino, asian, 
    american_indian_or_native_hawaiian_or_pacific_islander, multiracial
    """
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower().strip()  # Convert to lowercase and strip whitespace
    
    # Remove unwanted characters (commas, periods, etc.)
    x = re.sub(r'[",.\']', '', x)
    
    # Standardize different variations
    if any(keyword in x for keyword in ['white', 'caucasian', 'european']):
        return "white"
    elif any(keyword in x for keyword in ['black', 'african american', 'jamaican', 'haitian', 'ethiopian', 'somali']):
        return "black_or_african_american"
    elif any(keyword in x for keyword in ['hispanic', 'latino', 'mexican', 'puerto rican', 'salvadoran', 'cuban', 'dominican']):
        return "hispanic_or_latino"
    elif any(keyword in x for keyword in ['asian', 'chinese', 'indian', 'filipino', 'vietnamese', 'korean', 'japanese']):
        return "asian"
    elif any(keyword in x for keyword in ['native american', 'alaskan native', 'hawaiian', 'pacific islander', 'navajo', 'samoan']):
        return "american_indian_or_native_hawaiian_or_pacific_islander"
    elif any(keyword in x for keyword in ['multiracial', 'biracial', 'mixed-race', 'dual heritage']):
        return "multiracial"
    else:
        return np.nan  # In case there's an unmatched category

df_filtered["ethnicity_clean"] = df_filtered["Demo_ethnicity_Category"].apply(clean_ethnicity)
print(df_filtered["ethnicity_clean"].value_counts(dropna=False))

ethnicity_clean
white                        7088
black_or_african_american     564
hispanic_or_latino            376
multiracial                   273
asian                         162
Name: count, dtype: int64


In [14]:
state_abbreviation_mapping = {
    'alabama': 'AL',
    'alaska': 'AK',
    'arizona': 'AZ',
    'arkansas': 'AR',
    'california': 'CA',
    'colorado': 'CO',
    'connecticut': 'CT',
    'delaware': 'DE',
    'florida': 'FL',
    'georgia': 'GA',
    'hawaii': 'HI',
    'idaho': 'ID',
    'illinois': 'IL',
    'indiana': 'IN',
    'iowa': 'IA',
    'kansas': 'KS',
    'kentucky': 'KY',
    'louisiana': 'LA',
    'maine': 'ME',
    'maryland': 'MD',
    'massachusetts': 'MA',
    'michigan': 'MI',
    'minnesota': 'MN',
    'mississippi': 'MS',
    'missouri': 'MO',
    'montana': 'MT',
    'nebraska': 'NE',
    'nevada': 'NV',
    'new hampshire': 'NH',
    'new jersey': 'NJ',
    'new mexico': 'NM',
    'new york': 'NY',
    'north carolina': 'NC',
    'north dakota': 'ND',
    'ohio': 'OH',
    'oklahoma': 'OK',
    'oregon': 'OR',
    'pennsylvania': 'PA',
    'rhode island': 'RI',
    'south carolina': 'SC',
    'south dakota': 'SD',
    'tennessee': 'TN',
    'texas': 'TX',
    'utah': 'UT',
    'vermont': 'VT',
    'virginia': 'VA',
    'washington': 'WA',
    'west virginia': 'WV',
    'wisconsin': 'WI',
    'wyoming': 'WY',
    # Handle the "not specified" cases
    'usa (not a specific state)': 'No State',
    'usa (no specific state identified)': 'No State',
    'usa (the user is from the united states but does not specify a state)': 'No State',
    'usa (the user is from the united states but no specific state is identified)': 'No State',
    'if they do not specify a state, but are still from the united states of america, write "usa"': 'No State',
    'not from a state in the usa': 'No State',
    'not from a state in the usa,': 'No State',
    # Add any other variations you expect
}
df_filtered['state_cleaned'] = df_filtered['Demo_which_state_of_the_usa_do_they_live_in__Category'].apply(lambda x: state_abbreviation_mapping.get(x.strip().lower(), x.strip().capitalize()))
print(df_filtered['state_cleaned'].value_counts(dropna=False))


state_cleaned
CA              1025
NY               731
FL               585
TX               573
PA               366
                ... 
Connecticut,       1
Michigan,          1
Ny,                1
Tennessee,         1
Utah,              1
Name: count, Length: 118, dtype: int64


In [ ]:
import pandas as pd

# USPS state abbreviations mapping
STATE_MAPPING = {
    # Standard abbreviations
    'alabama': 'AL', 'alaska': 'AK', 'arizona': 'AZ', 'arkansas': 'AR', 'california': 'CA',
    'colorado': 'CO', 'connecticut': 'CT', 'delaware': 'DE', 'florida': 'FL', 'georgia': 'GA',
    'hawaii': 'HI', 'idaho': 'ID', 'illinois': 'IL', 'indiana': 'IN', 'iowa': 'IA',
    'kansas': 'KS', 'kentucky': 'KY', 'louisiana': 'LA', 'maine': 'ME', 'maryland': 'MD',
    'massachusetts': 'MA', 'michigan': 'MI', 'minnesota': 'MN', 'mississippi': 'MS', 'missouri': 'MO',
    'montana': 'MT', 'nebraska': 'NE', 'nevada': 'NV', 'new hampshire': 'NH', 'new jersey': 'NJ',
    'new mexico': 'NM', 'new york': 'NY', 'north carolina': 'NC', 'north dakota': 'ND', 'ohio': 'OH',
    'oklahoma': 'OK', 'oregon': 'OR', 'pennsylvania': 'PA', 'rhode island': 'RI', 'south carolina': 'SC',
    'south dakota': 'SD', 'tennessee': 'TN', 'texas': 'TX', 'utah': 'UT', 'vermont': 'VT',
    'virginia': 'VA', 'washington': 'WA', 'west virginia': 'WV', 'wisconsin': 'WI', 'wyoming': 'WY',
    'district of columbia': 'DC', 'washington dc': 'DC', 'dc': 'DC',
    'ny,':'NY',  'wa,': 'WA',
    # Handle common variations
    'california,': 'CA', 'new york,': 'NY', 'texas,': 'TX', 'florida,': 'FL', 'illinois,': 'IL',
    'massachusetts,': 'MA', 'michigan,': 'MI', 'minnesota,': 'MN', 'arizona,': 'AZ', 'nevada,': 'NV',
    'tennessee,': 'TN', 'south carolina,': 'SC', 'kansas,': 'KS', 'louisiana,': 'LA', 'maine,': 'ME',
    'connecticut,': 'CT', 'utah,': 'UT', 'idaho,': 'NV',
    
    # Handle parenthetical variations
    'california (ca)': 'CA', 'arizona (az)': 'AZ', 'kansas (ks)': 'KS', 'tennessee (tn)': 'TN',
    'hi (hawaii)': 'HI', 'ma (massachusetts)': 'MA', 'ca (california)': 'CA',
    
    # Handle USA cases
    'usa': 'USA', 'usa,': 'USA', 'usa (no specific state)': 'USA', 'usa (unspecified state)': 'USA',
    'if they do not specify a state, but are still from the united states of america, write "usa"': 'USA',
    'no state': 'USA'
}

def clean_state(state):
    """Convert state names to USPS abbreviations"""
    if pd.isna(state):
        return 'USA'
    
    state_str = str(state).strip().lower()
    
    # Direct mapping lookup
    if state_str in STATE_MAPPING:
        return STATE_MAPPING[state_str]
    
    # Handle case variations (Ca, Ny, Fl, etc.)
    if len(state_str) == 2 and state_str.upper() in STATE_MAPPING.values():
        return state_str.upper()
    
    # Handle common prefixes
    if state_str.startswith('new '):
        state_str = 'new ' + state_str[4:]
    elif state_str.startswith('north '):
        state_str = 'north ' + state_str[6:]
    elif state_str.startswith('south '):
        state_str = 'south ' + state_str[6:]
    elif state_str.startswith('west '):
        state_str = 'west ' + state_str[5:]
    
    # Try to find partial matches
    for key, value in STATE_MAPPING.items():
        if key in state_str:
            return value
    
    # If no match found, return original in uppercase
    return state_str.upper()

# Apply the cleaning function to your data
# Assuming your dataframe is called df and the column is 'state_cleaned'
df_filtered['state_cleaned'] = df_filtered['state_cleaned'].apply(clean_state)

print(len(df_filtered['state_cleaned'].value_counts()))

49


In [49]:
def clean_age(x):
    x = str(x).lower()
    if "18" in x or "24" in x:
        return "18-24"
    elif "25" in x or "34" in x:
        return "25-34"
    elif "35" in x or "44" in x:
        return "35-44"
    elif "45" in x or "54" in x:
        return "45-54"
    elif "55" in x or "64" in x:
        return "55-64"
    elif "65" in x or "older" in x:
        return "65+"
    else:
        return np.nan

df_filtered["age_group"] = df_filtered["Demo_age_Category"].apply(clean_age)
print(df_filtered['age_group'].value_counts())

age_group
35-44    2202
25-34    2003
55-64    1491
45-54    1384
65+       880
18-24     477
Name: count, dtype: int64


In [ ]:
print(len(df_filtered['state_cleaned'].value_counts()))

48


In [33]:
print(df_filtered['state_cleaned'].value_counts())

state_cleaned
CA    1203
NY     842
FL     625
TX     590
PA     383
WA     318
VA     278
MA     245
AZ     241
DC     236
NJ     207
OH     206
GA     201
CO     200
TN     196
IL     196
MN     195
MI     187
NC     185
OR     152
MD     142
NV     136
SC     134
WI     114
KS     100
UT      85
IA      85
LA      73
IN      71
KY      70
AL      64
CT      61
HI      51
DE      49
AR      48
MT      40
ID      35
NH      35
MO      29
NM      19
MS      18
WY      18
SD      16
OK      16
ME      12
RI      11
VT      11
NE       8
Name: count, dtype: int64


In [31]:
df_filtered = df_filtered[df_filtered['state_cleaned'] != 'USA']

In [7]:
df_filtered["y_ord"] = df_filtered["belief_clean"].map({
    "strongly_disagree": 1,
    "somewhat_disagree": 2,
    "somewhat_agree": 3,
    "strongly_agree": 4
})

In [ ]:
df_filtered = df_filtered.drop(['Demo_which_state_of_the_usa_do_they_live_in__Category', 'Belief_opinion_on_the_following_statement_it_should_be_acceptable_for_a_pregnant_woman_to_obtain_a_legal_abortion_if_a_woman_wants_it_for_any_reason_Category', 'Demo_ethnicity_Category'], axis=1)

In [76]:
df_filtered = df_filtered.drop(['Demo_interest_in_politics_Category'], axis=1)

In [77]:
df_filtered

,Unnamed: 0,id,time_window_start,username,Demo_marital_status_Category,Demo_partisan_loyalties_Category,Demo_household_income_bracket_Category,belief_clean,y_ord,ethnicity_clean,state_cleaned,gender_cleaned,age_group,education_cleaned,vote_cleaned,politics_interest_cleaned
0,40,1542629834953719810,2024-05-19 00:00:00+00:00,Badash1821,married currently legally married and living w...,strongly identifies with republicans,between 50000 and 75000 usd per year,strongly_disagree,1,white,OH,Female,35-44,High School,Trump,Highly interested
1,41,1542629834953719810,2024-06-16 00:00:00+00:00,Badash1821,married currently legally married and living w...,strongly identifies with republicans,between 50000 and 75000 usd per year,strongly_disagree,1,white,OH,Female,35-44,High School,Trump,Highly interested
2,43,1542629834953719810,2024-07-14 00:00:00+00:00,Badash1821,married currently legally married and living w...,strongly identifies with republicans,between 50000 and 75000 usd per year,strongly_disagree,1,white,OH,Female,35-44,High School,Trump,Highly interested
3,44,1542629834953719810,2024-07-28 00:00:00+00:00,Badash1821,"single never married, including those who are ...",independent closer to the democrats,between 25000 and 50000 usd per year,strongly_agree,4,white,OH,Female,35-44,High School,Biden,Highly interested
4,45,1542629834953719810,2024-08-11 00:00:00+00:00,Badash1821,married currently legally married and living w...,strongly identifies with republicans,between 50000 and 75000 usd per year,strongly_disagree,1,white,OH,Female,35-44,High School,Trump,Highly interested
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8458,23985,142827500,2024-08-25 00:00:00+00:00,ndrwwwww,"single never married, including those who are ...",independent closer to the democrats,between 50000 and 75000 usd per year,strongly_agree,4,white,IL,Male,25-34,University,Biden,Highly interested
8459,23986,142827500,2024-09-08 00:00:00+00:00,ndrwwwww,"single never married, including those who are ...",independent closer to the democrats,between 75000 and 100000 usd per year,strongly_agree,4,white,IL,Male,25-34,University,Biden,Moderately interested
8460,23987,142827500,2024-09-22 00:00:00+00:00,ndrwwwww,"single never married, including those who are ...",independent not closer to either party,between 50000 and 75000 usd per year,strongly_agree,4,white,IL,Male,25-34,University,Biden,Moderately interested
8461,23988,142827500,2024-10-06 00:00:00+00:00,ndrwwwww,"single never married, including those who are ...",independent not closer to either party,between 50000 and 75000 usd per year,strongly_agree,4,white,IL,Male,25-34,University,Did not vote,Moderately interested


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/mnt/c/Users/yuyue/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/abortion_result.csv')



# -----------------------------
# 2. Create AGE_GROUP
# -----------------------------
# Standardize age bins to match the model’s expected categories
def clean_age(x):
    x = str(x).lower()
    if "18" in x or "24" in x:
        return "18-24"
    elif "25" in x or "34" in x:
        return "25-34"
    elif "35" in x or "44" in x:
        return "35-44"
    elif "45" in x or "54" in x:
        return "45-54"
    elif "55" in x or "64" in x:
        return "55-64"
    elif "65" in x or "older" in x:
        return "65+"
    else:
        return np.nan

df["age_group"] = df["age"].apply(clean_age)
df = df.dropna(subset=["age_group"])



In [ ]:
#!/usr/bin/env python3
"""
run_mrp_icarrw1.py

Full pipeline:
 - Build adjacency from shapefile (states)
 - Prepare data from gss csv and poststrat csv
 - Compile and sample Stan model (ICAR for states, RW1 for age)
 - Post-stratify and plot state-level map.

Inputs (edit these paths / names to match your files):
  - GSS survey CSV: gss_csv
      required columns: y (0/1), sex, age_group, eth, state (2-letter USPS)
  - Post-stratification CSV: poststrat_csv
      required columns: sex, age_group, eth, state (2-letter USPS), N (population counts)
  - State shapefile (Cartographic Boundary shapefile directory or .shp path): states_shp_path
      e.g. 'cb_2018_us_state_500k.shp' or path to that shapefile
"""

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
from libpysal import weights
from scipy.linalg import eigh

# -------- USER EDIT: paths & column names --------
gss_csv = "gss.csv"                  # your survey data
poststrat_csv = "poststrat.csv"      # your stratification frame
states_shp_path = "cb_2018_us_state_500k.shp"  # cartographic states shapefile (or use tigris to download)
# column names in your csvs (change if different)
y_col = "y"               # 0/1 outcome
sex_col = "sex"
age_col = "age_group"     # ordered categorical variable (must match levels used in poststrat)
eth_col = "eth"
state_col = "state"       # 2-letter USPS code
post_N_col = "N"          # population counts in poststrat frame
# define the ordered age levels you want (string list). MUST match levels in both gss and poststrat.
age_levels = ["18-24","25-34","35-44","45-54","55-64","65+"]

# -------- End user-edit block --------

# 1) Load data
print("Loading data...")
gss = pd.read_csv(gss_csv)
post = pd.read_csv(poststrat_csv)

# Basic checks
required_gss_cols = {y_col, sex_col, age_col, eth_col, state_col}
required_post_cols = {sex_col, age_col, eth_col, state_col, post_N_col}
if not required_gss_cols.issubset(set(gss.columns)):
    raise ValueError(f"gss.csv missing required columns: {required_gss_cols - set(gss.columns)}")
if not required_post_cols.issubset(set(post.columns)):
    raise ValueError(f"poststrat.csv missing required columns: {required_post_cols - set(post.columns)}")

# Ensure age is categorical and ordered
gss[age_col] = pd.Categorical(gss[age_col], categories=age_levels, ordered=True)
post[age_col] = pd.Categorical(post[age_col], categories=age_levels, ordered=True)

# 2) Load shapefile (states) and subset to 50 states + DC (or as needed)
print("Reading shapefile...")
states_gdf = gpd.read_file(states_shp_path)

# Attempt to find USPS code column; common is 'STUSPS'
if 'STUSPS' in states_gdf.columns:
    shp_state_col = 'STUSPS'
elif 'STATEFP' in states_gdf.columns and 'STUSPS' in states_gdf.columns:
    shp_state_col = 'STUSPS'
else:
    # try common alternatives, else ask user to rename
    possible = [c for c in states_gdf.columns if states_gdf[c].dtype == object and states_gdf[c].str.len().max() <= 2]
    if len(possible) >= 1:
        shp_state_col = possible[0]
        print(f"Using shapefile column '{shp_state_col}' as state USPS code (confirm it's correct).")
    else:
        raise ValueError("Couldn't detect USPS code column in shapefile. Ensure shapefile has STUSPS column.")

# Keep only the 50 states + DC (match those present in post/gss)
keep_codes = sorted(set(list(pd.unique(gss[state_col])) + list(pd.unique(post[state_col]))))
# Filter shapefile to those codes (if present)
states_gdf = states_gdf[states_gdf[shp_state_col].isin(keep_codes)].copy()
states_gdf = states_gdf.reset_index(drop=True)
states_gdf[shp_state_col] = states_gdf[shp_state_col].astype(str)

# Create mapping from USPS code to index 1..S_state (Stan uses 1-based indices)
states_gdf['state_i'] = np.arange(1, len(states_gdf) + 1)
state_code_to_i = dict(zip(states_gdf[shp_state_col], states_gdf['state_i']))

# 3) Build neighbors using libpysal (contiguity - queen)
print("Building adjacency (queen contiguity)...")
# ensure consistent projection (project to an equal-area or projected CRS to avoid topology issues)
states_gdf = states_gdf.to_crs(epsg=5070)  # Albers Equal Area for US
w = weights.contiguity.Queen.from_dataframe(states_gdf)
# w.neighbors is dict: {index: [neigh indices (0-based)]}
# Build edges list (i<j unique pairs)
node_pairs = []
for i, neighs in w.neighbors.items():
    for j in neighs:
        if j > i:
            node_pairs.append((i + 1, j + 1))  # convert to 1-based indexing for Stan
node_pairs = np.array(node_pairs, dtype=int)
N_edges = node_pairs.shape[0]
print(f"Found {N_edges} edges among {len(states_gdf)} state polygons.")

# sumNumNeigh per node
sumNumNeigh = np.array([len(w.neighbors[i]) for i in range(len(states_gdf))], dtype=float)

# 4) compute ICAR scale factor
print("Computing ICAR scaling factor...")
# adjacency matrix (symmetric)
S = len(states_gdf)
A = np.zeros((S, S), dtype=float)
for i, neighs in w.neighbors.items():
    for j in neighs:
        A[i, j] = 1.0
        A[j, i] = 1.0
Q = np.diag(sumNumNeigh) - A
# eigenvalues; Q is symmetric
eigvals, eigvecs = eigh(Q)
# remove near-zero eigenvalues -> intrinsic CAR has at least one zero eigenvalue
nonzero = eigvals > 1e-8
if np.sum(nonzero) == 0:
    raise RuntimeError("All eigenvalues near zero - check adjacency matrix.")
scale_factor = np.mean(1.0 / eigvals[nonzero])
inv_sqrt_scale_factor = 1.0 / np.sqrt(scale_factor)
print(f"inv_sqrt_scale_factor = {inv_sqrt_scale_factor:.6f}")

# 5) Prepare model indices for observations
print("Preparing indices and filtering data to matching states...")
# map state codes in gss/post to numeric indices
gss = gss[gss[state_col].isin(state_code_to_i.keys())].copy()
gss['state_i'] = gss[state_col].map(state_code_to_i)
post = post[post[state_col].isin(state_code_to_i.keys())].copy()
post['state_i'] = post[state_col].map(state_code_to_i)

# Create integer-coded categories for sex, age, eth
gss['sex_i'] = pd.Categorical(gss[sex_col]).codes + 1
post['sex_i'] = pd.Categorical(post[sex_col], categories=pd.Categorical(gss[sex_col]).categories).codes + 1

gss['age_i'] = gss[age_col].cat.codes + 1
post['age_i'] = post[age_col].cat.codes + 1

gss['eth_i'] = pd.Categorical(gss[eth_col]).codes + 1
post['eth_i'] = pd.Categorical(post[eth_col], categories=pd.Categorical(gss[eth_col]).categories).codes + 1

# levels counts
S_sex = len(pd.Categorical(gss[sex_col]).categories)
S_age = len(age_levels)
S_eth = len(pd.Categorical(gss[eth_col]).categories)
S_state = S

print(f"Levels: sex={S_sex}, age={S_age}, eth={S_eth}, states={S_state}")
N = len(gss)

# 6) create the Stan model file (mrp_icarrw1.stan)
stan_code = r"""
data {
  int<lower=1> N;
  int<lower=0,upper=1> y[N];
  int<lower=1> S_sex;
  int<lower=1> S_age;
  int<lower=1> S_eth;
  int<lower=1> S_state;
  int<lower=1,upper=S_sex> sex_idx[N];
  int<lower=1,upper=S_age>  age_idx[N];
  int<lower=1,upper=S_eth>  eth_idx[N];
  int<lower=1,upper=S_state> state_idx[N];

  int<lower=0> N_edges;
  int<lower=1> node1[N_edges];
  int<lower=1> node2[N_edges];
  vector[S_state] sumNumNeigh;
  real<lower=0> inv_sqrt_scale_factor;
}

parameters {
  real alpha;
  vector[S_sex] z_sex;
  vector[S_eth] z_eth;
  vector[S_age] eta_age_raw;
  vector[S_state] u_raw;

  real<lower=0> sigma_sex;
  real<lower=0> sigma_eth;
  real<lower=0> sigma_age;
  real<lower=0> sigma_state;
}

transformed parameters {
  vector[S_sex] eta_sex = z_sex * sigma_sex;
  vector[S_eth] eta_eth = z_eth * sigma_eth;

  vector[S_age] eta_age;
  eta_age[1] = eta_age_raw[1] * sigma_age;
  for (k in 2:S_age)
    eta_age[k] = eta_age[k-1] + eta_age_raw[k] * sigma_age;

  vector[S_state] u = (inv_sqrt_scale_factor * u_raw) * sigma_state;
}

model {
  alpha ~ normal(0, 2.5);

  z_sex ~ normal(0,1);
  z_eth ~ normal(0,1);
  eta_age_raw ~ normal(0,1);

  sigma_sex ~ normal(0,1);
  sigma_eth ~ normal(0,1);
  sigma_age ~ normal(0,1);
  sigma_state ~ normal(0,1);

  // ICAR prior via pairwise differences:
  for (e in 1:N_edges) {
    real diff = u_raw[node1[e]] - u_raw[node2[e]];
    target += -0.5 * diff * diff;
  }

  for (i in 1:N) {
    real lin = alpha
               + eta_sex[sex_idx[i]]
               + eta_age[age_idx[i]]
               + eta_eth[eth_idx[i]]
               + u[state_idx[i]];
    y[i] ~ bernoulli_logit(lin);
  }
}

generated quantities {
  vector[S_state] state_effect = (inv_sqrt_scale_factor * u_raw) * sigma_state;
  vector[S_age] age_effect;
  vector[S_sex] sex_effect = z_sex * sigma_sex;
  vector[S_eth] eth_effect = z_eth * sigma_eth;
  age_effect[1] = eta_age_raw[1] * sigma_age;
  for (k in 2:S_age) age_effect[k] = age_effect[k-1] + eta_age_raw[k] * sigma_age;
}
"""

stan_file = "mrp_icarrw1.stan"
with open(stan_file, "w") as f:
    f.write(stan_code)
print(f"Wrote Stan model to {stan_file}")

# 7) Prepare Stan data dict
stan_data = {
    "N": int(N),
    "y": gss[y_col].astype(int).tolist(),
    "S_sex": int(S_sex),
    "S_age": int(S_age),
    "S_eth": int(S_eth),
    "S_state": int(S_state),
    "sex_idx": gss['sex_i'].astype(int).tolist(),
    "age_idx": gss['age_i'].astype(int).tolist(),
    "eth_idx": gss['eth_i'].astype(int).tolist(),
    "state_idx": gss['state_i'].astype(int).tolist(),
    "N_edges": int(N_edges),
    "node1": (node_pairs[:,0].astype(int)).tolist(),
    "node2": (node_pairs[:,1].astype(int)).tolist(),
    "sumNumNeigh": sumNumNeigh.tolist(),
    "inv_sqrt_scale_factor": float(inv_sqrt_scale_factor)
}

# 8) Compile and sample with cmdstanpy
print("Compiling Stan model (this may take a minute)...")
model = CmdStanModel(stan_file=stan_file)

print("Sampling...")
fit = model.sample(data=stan_data,
                   seed=123,
                   chains=4,
                   parallel_chains=4,
                   iter_warmup=1000,
                   iter_sampling=1000,
                   adapt_delta=0.95)

# Save fit
fit.save_csvfiles(dir="cmdstan_out")
print("Sampling complete. Summary:")
print(fit.summary().to_string())

# 9) Extract posterior draws (state_effect, age_effect, sex_effect, eth_effect, alpha)
print("Extracting draws for poststratification...")
# cmdstanpy: use stan_variable or stan_variables?
# We'll extract as numpy arrays using .draws_pd() or .stan_variable
state_effect_draws = fit.draws_pd(vars=['state_effect']).filter(regex=r'^state_effect\[').to_numpy()
# draws_pd returns a DataFrame of draws x variables; columns are like state_effect[1],...
# We want an array shape (n_draws, S_state)
n_draws = state_effect_draws.shape[0]
state_effect_draws = state_effect_draws.reshape(n_draws, S_state)

age_effect_draws = fit.draws_pd(vars=['age_effect']).filter(regex=r'^age_effect\[').to_numpy().reshape(n_draws, S_age)
sex_effect_draws = fit.draws_pd(vars=['sex_effect']).filter(regex=r'^sex_effect\[').to_numpy().reshape(n_draws, S_sex)
eth_effect_draws = fit.draws_pd(vars=['eth_effect']).filter(regex=r'^eth_effect\[').to_numpy().reshape(n_draws, S_eth)
alpha_draws = fit.draws_pd(vars=['alpha'])['alpha'].to_numpy()

print(f"Posterior draws: {n_draws} draws.")

# 10) Post-stratify
print("Post-stratifying to states...")
# Prepare post frame rows
post = post.copy()
post['cell_id'] = np.arange(len(post))
# ensure that all category levels are aligned and that indices were created earlier
# linear predictor per draw for each post cell: alpha + sex + age + eth + state
# we will compute for each draw, then aggregate weighted by N per state

# Convert post indices to zero-based indices for numpy indexing (but our draw arrays are 0-based)
post_sex_idx = post['sex_i'].astype(int) - 1
post_age_idx = post['age_i'].astype(int) - 1
post_eth_idx = post['eth_i'].astype(int) - 1
post_state_idx = post['state_i'].astype(int) - 1
post_N = post[post_N_col].astype(float).to_numpy()

# pre-allocate draws x states matrix
state_probs = np.zeros((n_draws, S_state), dtype=float)

for d in range(n_draws):
    # linear predictor per cell
    lin = (alpha_draws[d]
           + sex_effect_draws[d, post_sex_idx]
           + age_effect_draws[d, post_age_idx]
           + eth_effect_draws[d, post_eth_idx]
           + state_effect_draws[d, post_state_idx])
    p = 1.0 / (1.0 + np.exp(-lin))
    # weighted mean per state
    # sum p*N by state index
    for s in range(S_state):
        mask = (post_state_idx == s)
        if mask.sum() == 0:
            state_probs[d, s] = np.nan
        else:
            num = np.sum(p[mask] * post_N[mask])
            den = np.sum(post_N[mask])
            state_probs[d, s] = num / den

# posterior summaries
state_mean = np.nanmean(state_probs, axis=0)
state_sd = np.nanstd(state_probs, axis=0)

# attach to shapefile GeoDataFrame
states_gdf = states_gdf.sort_values('state_i').reset_index(drop=True)
states_gdf['pred_mean'] = state_mean
states_gdf['pred_sd'] = state_sd

# 11) Plot choropleth
print("Plotting map...")
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
states_gdf.plot(column='pred_mean', ax=ax, legend=True, cmap='viridis',
                legend_kwds={'shrink': 0.6}, edgecolor='0.2', linewidth=0.4)
ax.set_title("MrP predicted probability (state-level) — post-stratified")
ax.axis('off')
plt.tight_layout()
plt.savefig("state_mrp_map.png", dpi=150)
print("Saved map to state_mrp_map.png")

# 12) Save posterior summaries to CSV
states_gdf[['state_i', shp_state_col, 'pred_mean', 'pred_sd']].to_csv("state_post_pred_summary.csv", index=False)
print("Saved state posterior summary to state_post_pred_summary.csv")

print("Finished.")
